# detecting.expr3.frkern.fixed — Rh/Rl table (rkern × fkern)

**Data source**: `detecting.expr3.frkern.fixed` runs produced by [`TacVar/scripts/run_detecting_expr3_frkern_fixed10000_0416.sh`](../scripts/run_detecting_expr3_frkern_fixed10000_0416.sh).

This notebook scans all combos under a single `TIMESTAMP`, computes **Rh/Rl** (TacVar v0.0 tile metrics) for each **timer × (rkern,fkern)**, then tabulates a **rkern × fkern** table.

- **Rows**: `rkern`
- **Columns**: `fkern`
- **Output**: one table **per timer**. Rows=`rkern`. Columns are two-level: top=`fkern`, subcolumns=`Rh` and `Rl`.

Metric definition:
- `U`: measured `ta` samples (`partes_ta_r*.csv`) for the combo (expects **single walk**: `NWALKS=1`).
- `V`: theory samples from `Normal(mu_ns, SIGMA_REL × mu_ns)`.
- `Rh/Rl`: `TacVar/utils/metrics_wasserstein.py:w_rl_rh_from_samples(U, V, ntiles=100, q=0.9, drop=0.995)`.

This notebook prints **one table per timer**.

Adjust **cell 1** (`DATA_ROOT`, `HOSTNAME`, `TIMESTAMP`, optional combo filters).


In [21]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "TacVar" / "utils" / "metrics_wasserstein.py").is_file():
            return p
    raise FileNotFoundError("Could not locate repo root containing TacVar/utils/metrics_wasserstein.py")


_REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from TacVar.utils.metrics_wasserstein import w_rl_rh_from_samples

# =========================
# Config
# =========================

DATA_ROOT = Path("/astrum/home/hpchzy/code/data")

DATETIME = "20260416"
HOSTNAME = "cgnr6760pn2"
EXPR_NAME = "detecting.expr3.frkern.fixed"
TIMESTAMP = "20260416_151903" # 原方法，不锁频率

EXP_ROOT = Path(f"{DATA_ROOT}/{DATETIME}/output/{HOSTNAME}/{EXPR_NAME}/{TIMESTAMP}")
print("EXP_ROOT:", EXP_ROOT.resolve())

WALK_LIST_NAME = "walk_list_normal.csv"

# Optional combo filters (set to None to accept all)
INTERVAL_NS: int | None = 10000
FSIZE_KIB: int | None = 4096
RSIZE_KIB: int | None = None

# Theory
SIGMA_REL = 0.015
N_THEORY = 50_000
SEED = 42

# Metric parameters
NTILES_METRIC = 100
Q_SPLIT = 0.9
DROP = 0.995
DENOM_POLICY = "inf"

# Per-timer tables will be printed (no timer aggregation).
# You can optionally restrict timers here (e.g. {'clock_gettime','papi'}); set None to keep all.
TIMER_WHITELIST: set[str] | None = None

EXP_ROOT: /astrum/home/hpchzy/code/data/20260416/output/cgnr6760pn2/detecting.expr3.frkern.fixed/20260416_151903


In [22]:
def _parse_combo(combo_name: str) -> dict[str, str]:
    out: dict[str, str] = {}
    for part in combo_name.split("_"):
        if not part:
            continue
        for k in ("interval", "fkern", "rkern", "fsize", "rsize"):
            if part.startswith(k):
                out[k] = part[len(k) :]
                break
    return out


def combo_dir_matches(
    combo_name: str,
    *,
    interval_ns: int | None,
    fsize_kib: int | None,
    rsize_kib: int | None,
) -> bool:
    info = _parse_combo(combo_name)
    if interval_ns is not None and int(info.get("interval", "-1")) != int(interval_ns):
        return False
    if fsize_kib is not None and int(info.get("fsize", "-1")) != int(fsize_kib):
        return False
    if rsize_kib is not None:
        parsed = info.get("rsize")
        if parsed is None:
            return int(rsize_kib) == 0
        if int(parsed) != int(rsize_kib):
            return False
    return True


def list_timers(exp_root: Path) -> list[str]:
    timers: list[str] = []
    for p in sorted(exp_root.iterdir()):
        if not p.is_dir() or p.name.startswith("."):
            continue
        if p.name == "_archive":
            continue
        for combo in p.iterdir():
            if not combo.is_dir():
                continue
            walks = combo / f"{p.name}_walks"
            if walks.is_dir():
                timers.append(p.name)
                break
    return timers


def load_experiment_meta(meta_dir: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    p_md = meta_dir / "meta.md"
    if p_md.is_file():
        for m in re.finditer(
            r"^\|\s*([^|\n]+?)\s*\|\s*`([^`]*)`\s*\|",
            p_md.read_text(),
            re.MULTILINE,
        ):
            k = m.group(1).strip()
            if not k or k in ("Key", "Parameter", "Item") or k.startswith("---"):
                continue
            out[k] = m.group(2).strip()
    return out


def read_walk_dataframe(exp_root: Path, combo_dir: Path) -> tuple[pd.DataFrame, Path]:
    p_root = exp_root / WALK_LIST_NAME
    if p_root.is_file():
        return pd.read_csv(p_root), p_root
    p_combo = combo_dir / WALK_LIST_NAME
    if p_combo.is_file():
        return pd.read_csv(p_combo), p_combo
    raise FileNotFoundError(f"No {WALK_LIST_NAME} under {exp_root} or {combo_dir}")


def assert_single_walk(walk_df: pd.DataFrame) -> None:
    if len(walk_df.index) != 1:
        raise ValueError("Expected single-walk run (NWALKS=1)")


def resolve_mu_ns(combo_path: Path, walk_df: pd.DataFrame) -> float:
    info = _parse_combo(combo_path.name)
    if "interval" in info:
        return float(info["interval"])
    meta = load_experiment_meta(combo_path)
    for k in ("walk_mu_ns", "walk_fixed_ns", "interval_ns"):
        v = meta.get(k)
        if v:
            return float(v)
    return float(walk_df["ta_ns"].iloc[0])


def single_walk_dir(timer: str, combo: Path) -> Path:
    root = combo / f"{timer}_walks"
    dirs = sorted(p for p in root.glob("w*") if p.is_dir())
    if len(dirs) != 1:
        raise ValueError(f"Expected 1 walk dir under {root}, got {len(dirs)}")
    return dirs[0]


def load_ta_samples(walk_dir: Path) -> np.ndarray:
    blocks: list[np.ndarray] = []
    for f in sorted(walk_dir.glob("partes_ta_r*.csv")):
        arr = np.loadtxt(f, dtype=np.int64)
        if arr.ndim == 0:
            arr = np.array([arr], dtype=np.int64)
        blocks.append(arr.ravel())
    return np.concatenate(blocks, axis=0) if blocks else np.array([], dtype=np.int64)


@dataclass(frozen=True)
class PointResult:
    timer: str
    rkern: str
    fkern: str
    interval_ns: int
    fsize_kib: int
    mu_ns: float
    n_meas: int
    rl: float
    rh: float


In [23]:
# =========================
# Scan combos and compute Rh/Rl per timer×(rkern,fkern)
# =========================

exp_root = Path(EXP_ROOT).resolve()
if not exp_root.is_dir():
    raise FileNotFoundError(f"EXP_ROOT not found: {exp_root}")

timers = list_timers(exp_root)
if not timers:
    raise RuntimeError(f"No timer directories under {exp_root}")
if TIMER_WHITELIST is not None:
    timers = [t for t in timers if t in TIMER_WHITELIST]
    if not timers:
        raise RuntimeError(f"No timers left after TIMER_WHITELIST={TIMER_WHITELIST}")

rows: list[PointResult] = []
for tm in timers:
    tdir = exp_root / tm
    if not tdir.is_dir():
        continue
    for combo in sorted(tdir.iterdir()):
        if not combo.is_dir():
            continue
        if not combo_dir_matches(combo.name, interval_ns=INTERVAL_NS, fsize_kib=FSIZE_KIB, rsize_kib=RSIZE_KIB):
            continue
        info = _parse_combo(combo.name)
        fk = info.get("fkern")
        rk = info.get("rkern")
        ival = info.get("interval")
        fs = info.get("fsize")
        if fk is None or rk is None or ival is None or fs is None:
            continue

        walk_df, _ = read_walk_dataframe(exp_root, combo)
        assert_single_walk(walk_df)
        mu_ns = resolve_mu_ns(combo, walk_df)
        sigma_abs = float(mu_ns) * float(SIGMA_REL)

        wd = single_walk_dir(tm, combo)
        U = load_ta_samples(wd).astype(np.float64)
        if U.size == 0:
            continue

        # Deterministic per (rkern,fkern)
        seed = int(SEED) + (hash(rk) & 0xFFFF) * 131 + (hash(fk) & 0xFFFF)
        rng = np.random.default_rng(seed)
        V = rng.normal(float(mu_ns), float(sigma_abs), size=int(N_THEORY)).astype(np.float64)

        res = w_rl_rh_from_samples(
            U,
            V,
            ntiles=int(NTILES_METRIC),
            q=float(Q_SPLIT),
            drop=float(DROP),
            denom_policy=str(DENOM_POLICY),
        )
        rows.append(
            PointResult(
                timer=tm,
                rkern=str(rk),
                fkern=str(fk),
                interval_ns=int(ival),
                fsize_kib=int(fs),
                mu_ns=float(mu_ns),
                n_meas=int(U.size),
                rl=float(res["rl"]),
                rh=float(res["rh"]),
            )
        )

df = pd.DataFrame([r.__dict__ for r in rows])
if df.empty:
    raise RuntimeError("No data points computed (filters too strict, or missing outputs)")

display(df.head(20))
print("n_rows:", len(df.index), "timers:", sorted(df["timer"].unique().tolist()))

,timer,rkern,fkern,interval_ns,fsize_kib,mu_ns,n_meas,rl,rh
0,clock_gettime,add,add,10000,4096,10000.00,6400,1.85,0.70
1,clock_gettime,copy,add,10000,4096,10000.00,6400,1.47,0.87
2,clock_gettime,pow,add,10000,4096,10000.00,6400,1.51,0.87
3,clock_gettime,scale,add,10000,4096,10000.00,6400,1.77,0.17
4,clock_gettime,triad,add,10000,4096,10000.00,6400,1.76,0.74
5,clock_gettime,add,copy,10000,4096,10000.00,6400,1.69,0.10
6,clock_gettime,copy,copy,10000,4096,10000.00,6400,1.59,0.64
7,clock_gettime,pow,copy,10000,4096,10000.00,6400,1.51,0.77
8,clock_gettime,scale,copy,10000,4096,10000.00,6400,1.61,0.83
9,clock_gettime,triad,copy,10000,4096,10000.00,6400,1.54,0.13


n_rows: 150 timers: ['clock_gettime', 'likwid', 'mpi_wtime', 'papi', 'papix6', 'tsc_asym']


In [24]:
# =========================
# Tabulate per-timer: rows=rkern, columns=(fkern, {Rh,Rl})
# =========================

# Display formatting: 2 decimal places
pd.options.display.float_format = "{:.2f}".format

for tm in sorted(df["timer"].unique().tolist()):
    d = df[df["timer"] == tm].copy()

    # Build a 2-level column table. We want: top level = fkern, second level = metric (Rh/Rl)
    tbl = d.pivot(index="rkern", columns="fkern", values=["rh", "rl"])
    tbl = tbl.swaplevel(0, 1, axis=1)  # (fkern, metric)
    tbl = tbl.sort_index(axis=1, level=0)

    # Order the metric sub-columns and rename to Rh/Rl
    fkerns = sorted(tbl.columns.get_level_values(0).unique().tolist())
    metric_order = ["rh", "rl"]
    cols = [(fk, m) for fk in fkerns for m in metric_order if (fk, m) in tbl.columns]
    tbl = tbl.loc[:, cols]
    tbl = tbl.rename(columns={"rh": "Rh", "rl": "Rl"}, level=1)

    tbl = tbl.reindex(sorted(tbl.index), axis=0).round(2)

    print(f"timer={tm}")
    display(tbl)


timer=clock_gettime


fkern  add      copy       pow      scale      triad     
        Rh   Rl   Rh   Rl   Rh   Rl    Rh   Rl    Rh   Rl
rkern                                                    
add   0.70 1.85 0.10 1.69 0.18 1.43  0.40 1.58  0.68 1.79
copy  0.87 1.47 0.64 1.59 0.23 1.56  0.63 1.56  0.36 1.86
pow   0.87 1.51 0.77 1.51 0.77 1.83  0.16 1.57  0.09 1.49
scale 0.17 1.77 0.83 1.61 0.77 1.55  0.74 1.48  0.35 1.83
triad 0.74 1.76 0.13 1.54 0.28 1.41  0.18 1.47  0.90 1.72

timer=likwid


fkern  add      copy       pow      scale      triad     
        Rh   Rl   Rh   Rl   Rh   Rl    Rh   Rl    Rh   Rl
rkern                                                    
add   0.73 1.59 0.27 1.67 0.65 1.36  0.10 1.66  0.67 1.62
copy  0.71 1.66 0.18 1.79 0.09 1.66  0.76 1.65  0.25 1.53
pow   0.23 1.67 0.78 1.55 0.23 1.61  0.73 1.52  0.72 1.55
scale 0.26 1.85 0.71 1.55 0.23 1.44  0.12 1.68  0.20 1.53
triad 0.73 1.79 0.11 1.58 0.11 1.57  0.30 1.60  0.26 1.89

timer=mpi_wtime


...

fkern  add      copy       pow      scale      triad     
        Rh   Rl   Rh   Rl   Rh   Rl    Rh   Rl    Rh   Rl
rkern                                                    
add   0.84 2.34 0.73 2.55 0.67 3.63  0.76 2.25  0.87 2.45
copy  0.73 2.30 0.76 2.48 0.75 3.31  0.73 2.27  0.75 2.48
pow   0.87 2.23 0.77 2.28 0.56 2.86  0.77 2.20  0.77 2.30
scale 0.80 2.24 0.73 2.30 0.52 3.34  0.87 2.39  0.85 2.28
triad 0.86 2.54 0.86 2.30 0.74 3.45  0.72 2.29  0.86 2.32

timer=papix6


fkern  add      copy       pow      scale      triad     
        Rh   Rl   Rh   Rl   Rh   Rl    Rh   Rl    Rh   Rl
rkern                                                    
add   0.72 2.35 0.91 2.43 0.50 2.94  0.85 2.38  0.71 2.44
copy  0.74 2.31 0.73 2.20 0.49 3.67  0.74 2.33  0.91 2.22
pow   0.78 2.15 0.78 2.33 0.76 3.09  0.88 2.11  0.77 2.39
scale 0.87 2.26 0.75 2.17 0.75 3.02  0.76 2.27  0.65 2.37
triad 0.71 2.40 0.91 2.46 0.73 3.80  0.68 2.37  0.84 2.34

timer=tsc_asym


fkern  add      copy       pow      scale      triad     
        Rh   Rl   Rh   Rl   Rh   Rl    Rh   Rl    Rh   Rl
rkern                                                    
add   0.67 1.30 0.31 1.56 0.57 1.51  0.27 1.48  0.21 1.57
copy  0.73 1.25 0.27 1.46 0.63 1.35  0.44 1.36  0.35 1.27
pow   0.34 1.30 0.70 1.37 0.63 1.34  0.71 1.31  0.75 1.31
scale 0.73 1.47 0.40 1.47 0.54 1.32  0.72 1.23  0.23 1.40
triad 0.23 1.45 0.22 1.40 0.60 1.32  0.24 1.32  0.35 1.58